# 03 Data Cleaning & Type Casting
**Project:** Retailmart Finance Analytics  
**Phase:** Handle missing values, cast datatypes (like dates), remove duplicates, and export cleaned files.

---

### 1. Setup & Imports
We must add the parent directory to the system path so Python can locate the custom `src` modules.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add root directory to python path
sys.path.append(os.path.abspath(os.path.join('..')))

# Import custom configurations and utilities
from src.config.config import config
from src.database.connection import db_connector
from src.utils.logger import setup_logger
from src.utils.helpers import save_dataframe

logger = setup_logger(os.path.splitext(os.path.basename('__file__'))[0])
logger.info('Notebook setup and imports loaded successfully.')

[2026-07-17 14:20:49] [INFO] [config.py:16] Loaded environment configuration from: E:\SQL ACCIO JOB\Cross functional dashboard\.env


[2026-07-17 14:20:49] [INFO] [connection.py:42] SQLAlchemy Database Engine initialized successfully.


[2026-07-17 14:20:49] [INFO] [4206973462.py:16] Notebook setup and imports loaded successfully.


### 2. Load Profiled Datasets
Placeholder for load profiled datasets steps.

In [2]:
import glob
import os

# Find all raw CSV files we extracted in Phase 2
raw_files = glob.glob("../data/raw/*.csv")
print(f"Found {len(raw_files)} raw files to clean:\n")

# Load into our dfs dictionary
dfs = {}
for file in raw_files:
    name = os.path.splitext(os.path.basename(file))[0]
    dfs[name] = pd.read_csv(file)
    print(f" - Loaded: {name} (Shape: {dfs[name].shape})")

Found 18 raw files to clean:

 - Loaded: customer_customers (Shape: (5000, 19))
 - Loaded: dealer_dealers (Shape: (200, 16))
 - Loaded: dealer_network_dealer_commissions (Shape: (3000, 9))
 - Loaded: dealer_network_dealer_targets (Shape: (4800, 8))
 - Loaded: dealer_network_regional_offices (Shape: (20, 4))
 - Loaded: finance_accounting_dealer_payouts (Shape: (2500, 7))
 - Loaded: finance_accounting_gl_accounts (Shape: (80, 6))
 - Loaded: finance_accounting_ledger_entries (Shape: (15000, 10))
 - Loaded: finance_accounting_tax_records (Shape: (7000, 8))
 - Loaded: location_reference_cities (Shape: (200, 7))
 - Loaded: location_reference_states (Shape: (36, 6))
 - Loaded: pricing_management_price_master (Shape: (1440, 10))
 - Loaded: product_management_vehicle_models (Shape: (25, 9))
 - Loaded: product_management_vehicle_variants (Shape: (96, 11))
 - Loaded: sales_transaction_invoices (Shape: (7000, 14))
 - Loaded: sales_transaction_orders (Shape: (7000, 16))
 - Loaded: sales_transaction

### 3. Handle Missing Values
Placeholder for handle missing values steps.

In [3]:
# 1. Clean 'sales_transaction_orders' null values
orders_df = dfs.get("sales_transaction_orders")
if orders_df is not None:
    # Fill empty finance providers with 'Self-Financed'
    orders_df["finance_provider"] = orders_df["finance_provider"].fillna("Self-Financed")
    # Fill empty discount amounts with 0.0 (prevents math addition errors later)
    orders_df["discount_amount"] = orders_df["discount_amount"].fillna(0.0)
    # Fill empty discount IDs with -1 (meaning no discount applied)
    orders_df["discount_id"] = orders_df["discount_id"].fillna(-1).astype(int)
    print("Filled null values in 'sales_transaction_orders'")

# 2. Clean 'sales_transaction_payments' null values
payments_df = dfs.get("sales_transaction_payments")
if payments_df is not None:
    if "finance_provider" in payments_df.columns:
        payments_df["finance_provider"] = payments_df["finance_provider"].fillna("Self-Financed")
    print("Filled null values in 'sales_transaction_payments'")

Filled null values in 'sales_transaction_orders'
Filled null values in 'sales_transaction_payments'


### 4. Data Type Conversions (e.g. DateTime)
Placeholder for data type conversions (e.g. datetime) steps.

In [4]:
# Convert date columns from plain text strings to proper pandas Datetime objects
date_cols_map = {
    "sales_transaction_orders": ["order_date", "created_at"],
    "sales_transaction_payments": ["payment_date", "created_at"],
    "sales_transaction_invoices": ["invoice_date", "created_at"],
    "finance_accounting_ledger_entries": ["entry_date", "created_at"],
    "finance_accounting_dealer_payouts": ["payout_date", "created_at"],
    "finance_accounting_tax_records": ["tax_date", "created_at"],
    "dealer_network_dealer_targets": ["target_month", "created_at"]
}

for df_name, date_cols in date_cols_map.items():
    df = dfs.get(df_name)
    if df is not None:
        for col in date_cols:
            if col in df.columns:
                # Convert to datetime specifying format='mixed' and dayfirst=True for mixed formats
                df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True, format='mixed')
        print(f"Casted date columns in '{df_name}' to datetime")

Casted date columns in 'sales_transaction_orders' to datetime
Casted date columns in 'sales_transaction_payments' to datetime


Casted date columns in 'sales_transaction_invoices' to datetime
Casted date columns in 'finance_accounting_ledger_entries' to datetime
Casted date columns in 'finance_accounting_dealer_payouts' to datetime
Casted date columns in 'finance_accounting_tax_records' to datetime
Casted date columns in 'dealer_network_dealer_targets' to datetime


### 5. Deduplication
Placeholder for deduplication steps.

In [5]:
# Detect and drop any duplicate rows across all 18 datasets
for name, df in dfs.items():
    duplicates_count = df.duplicated().sum()
    if duplicates_count > 0:
        dfs[name] = df.drop_duplicates()
        print(f" - Dataset '{name}': Removed {duplicates_count} duplicate rows.")
    else:
        print(f" - Dataset '{name}': No duplicates found.")

 - Dataset 'customer_customers': No duplicates found.
 - Dataset 'dealer_dealers': No duplicates found.
 - Dataset 'dealer_network_dealer_commissions': No duplicates found.
 - Dataset 'dealer_network_dealer_targets': No duplicates found.
 - Dataset 'dealer_network_regional_offices': No duplicates found.
 - Dataset 'finance_accounting_dealer_payouts': No duplicates found.
 - Dataset 'finance_accounting_gl_accounts': No duplicates found.
 - Dataset 'finance_accounting_ledger_entries': No duplicates found.
 - Dataset 'finance_accounting_tax_records': No duplicates found.
 - Dataset 'location_reference_cities': No duplicates found.
 - Dataset 'location_reference_states': No duplicates found.
 - Dataset 'pricing_management_price_master': No duplicates found.
 - Dataset 'product_management_vehicle_models': No duplicates found.
 - Dataset 'product_management_vehicle_variants': No duplicates found.
 - Dataset 'sales_transaction_invoices': No duplicates found.
 - Dataset 'sales_transaction_orde

### 6. Export Cleaned Datasets to 'data/processed/'
Placeholder for export cleaned datasets to 'data/processed/' steps.

In [6]:
# Export the cleaned datasets as CSVs to 'data/processed/'
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

success_count = 0
for name, df in dfs.items():
    filepath = os.path.join(processed_dir, f"{name}.csv")
    success = save_dataframe(df, filepath)
    if success:
        success_count += 1
        
print(f"\nData cleaning complete! Saved {success_count}/{len(dfs)} clean CSVs to: '{processed_dir}'")

[2026-07-17 14:20:50] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (5000, 19) to: ../data/processed\customer_customers.csv


[2026-07-17 14:20:50] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (200, 16) to: ../data/processed\dealer_dealers.csv


[2026-07-17 14:20:50] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (3000, 9) to: ../data/processed\dealer_network_dealer_commissions.csv


[2026-07-17 14:20:50] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (4800, 8) to: ../data/processed\dealer_network_dealer_targets.csv


[2026-07-17 14:20:50] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (20, 4) to: ../data/processed\dealer_network_regional_offices.csv


...

[2026-07-17 14:20:51] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (7000, 14) to: ../data/processed\sales_transaction_invoices.csv


[2026-07-17 14:20:51] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (7000, 16) to: ../data/processed\sales_transaction_orders.csv


[2026-07-17 14:20:51] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (7500, 12) to: ../data/processed\sales_transaction_order_items.csv


[2026-07-17 14:20:51] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (8000, 11) to: ../data/processed\sales_transaction_payments.csv



Data cleaning complete! Saved 18/18 clean CSVs to: '../data/processed'
